# Extra de S5 · MCP en notebooks · 1. Un MCP que consulta la capa master (SQLite y BigQuery)

**Qué vas a construir:** un servidor MCP que deja a un LLM hacer preguntas sobre los clientes, facturas y productos del caso Retail, **sin darle acceso libre a la base de datos**.

**Qué vas a aprender:**
- cómo se publica una consulta como *herramienta* con nombre, argumentos y una descripción que el LLM lee;
- por qué un «SQL libre» es peligroso y cómo se vigila en **capas** (herramientas fijas, vigilancia del SQL, conexión de solo lectura, topes de filas y de coste);
- cómo un mismo servidor habla con **dos motores** (SQLite y BigQuery) escondiéndolos tras un *conector*, y cómo se **comprueba que dan lo mismo**.

> Un MCP (Model Context Protocol) es un protocolo estándar para que un LLM use herramientas de un servidor sin ver su código ni su base de datos.
> En el laboratorio de agentes de S5 ya usaste `mcp-retencion` (lista de clientes y puntuación). Aquí construyes los tuyos desde cero.

In [1]:
import asyncio, json, os, sys, sqlite3
from pathlib import Path

AQUI = Path.cwd()
RAIZ = next(p for p in [AQUI, *AQUI.parents] if (p / '04-Proyecto-Base').is_dir())    # funciona en tu portátil y en un Codespace
PROYECTO = RAIZ / '04-Proyecto-Base'
BD = PROYECTO / 'data' / 'processed' / 'curso.db'
TRABAJO = AQUI / '_trabajo'
TRABAJO.mkdir(exist_ok=True)
print('Repositorio:', RAIZ.name, '| base de datos:', BD.name, 'existe' if BD.is_file() else 'NO EXISTE (mira el notebook 00)')

Repositorio: curso-ml-ia-desarrollo-produccion | base de datos: curso.db existe


## 1. El servidor

Lee el código con calma: es la parte importante. Cuatro ideas:

1. **Conectores** (`ConectorSQLite`, `ConectorBigQuery`): ofrecen `tabla()`, `columnas()` y `ejecutar()`. El resto del servidor no sabe qué motor hay detrás.
2. **Herramientas con nombre** (`resumen_cliente`, `ventas_por_pais`...): el SQL lo escribes tú, el LLM solo da un número. Es lo más seguro.
3. **`consulta_sql`**: el último recurso, vigilado por `validar_sql` (solo SELECT, una sentencia, sin comentarios, solo tablas `master_*`).
4. **Las capas que no dependen de nadie**: SQLite se abre en modo solo lectura (`mode=ro`), BigQuery tiene un tope de bytes (`maximum_bytes_billed`) y nunca se devuelven más de 50 filas.

La celda siguiente **escribe el servidor en el archivo `_trabajo/mcp_datos.py`** (así lo puede lanzar un proceso aparte, que es lo que hacen los clientes MCP).

In [2]:
%%writefile _trabajo/mcp_datos.py
"""Servidor MCP «mcp-datos-master»: deja que un LLM consulte la capa master del caso Retail, en SQLite o en BigQuery.

POR QUÉ NO LE DAMOS «UN SQL LIBRE» SIN MÁS
    Un LLM que escribe SQL contra tu base de datos es potente y peligroso. Aquí hay cuatro capas, de fuera adentro:
      1. Herramientas con nombre y argumentos tipados (resumen_cliente, ventas_por_pais...): lo normal es no escribir SQL.
      2. Una herramienta de SQL libre (consulta_sql) pero VIGILADA: solo SELECT, una sentencia, sin comentarios, solo tablas master_*.
      3. La conexión solo puede leer: SQLite se abre en modo «ro»; BigQuery tiene un tope de bytes facturables por consulta.
      4. Un tope de filas devueltas (MAX_FILAS): el LLM no necesita 22 000 líneas, y su ventana de contexto tampoco.
    Las capas 3 y 4 no dependen del LLM ni de la vigilancia de la capa 2: si esta falla, siguen protegiendo.

DOS CONECTORES, LA MISMA INTERFAZ
    ConectorSQLite     lee curso.db (por defecto).            MCP_BACKEND=sqlite
    ConectorBigQuery   lee un dataset de BigQuery.            MCP_BACKEND=bigquery  GCP_PROJECT=...  BQ_DATASET=curso_master
    Ambos ofrecen tabla(), columnas() y ejecutar(). Las consultas usan parámetros con @nombre (valen para los dos motores).
"""
import json
import os
import re
import sqlite3
import sys
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path

from mcp.server.fastmcp import FastMCP

NOMBRE = 'mcp-datos-master'
TABLAS = ['master_clientes', 'master_facturas', 'master_lineas', 'master_productos']   # lo único que se puede consultar
MAX_FILAS = 50
MAX_BYTES_BQ = int(os.environ.get('MCP_BQ_MAX_BYTES', 100_000_000))                   # 100 MB por consulta: el dataset entero pesa ~2 MB


def _limpio(v):
    """Lo que devuelve una base de datos, a tipos que viajan por JSON."""
    if isinstance(v, Decimal):
        return float(v)
    if isinstance(v, (date, datetime)):
        return v.isoformat()
    return v


# ------------------------------------------------------------------------------------------------ conectores
class ConectorSQLite:
    nombre = 'sqlite'

    def __init__(self, ruta):
        self.ruta = Path(ruta)
        if not self.ruta.is_file():
            raise FileNotFoundError(f'No existe {self.ruta.name}: crea la base con  python -m ia_mlops.pipeline  (desde 04-Proyecto-Base)')

    def tabla(self, t):
        return t

    def _abrir(self):
        con = sqlite3.connect(self.ruta.resolve().as_uri() + '?mode=ro', uri=True)      # SOLO LECTURA: un UPDATE falla aunque se cuele
        con.row_factory = sqlite3.Row
        return con

    def ejecutar(self, sql, params=None):
        con = self._abrir()
        try:
            filas = [dict(f) for f in con.execute(sql, params or {}).fetchmany(MAX_FILAS + 1)]
        finally:
            con.close()
        return [{k: _limpio(v) for k, v in f.items()} for f in filas]

    def columnas(self, t):
        con = self._abrir()
        try:
            return [{'nombre': r[1], 'tipo': r[2] or 'sin tipo'} for r in con.execute(f'PRAGMA table_info("{t}")')]
        finally:
            con.close()


class ConectorBigQuery:
    nombre = 'bigquery'

    def __init__(self, proyecto, dataset):
        from google.cloud import bigquery
        self.bq = bigquery
        self.proyecto, self.dataset = proyecto, dataset
        self.cliente = bigquery.Client(project=proyecto)

    def tabla(self, t):
        return f'`{self.proyecto}.{self.dataset}.{t}`'

    def ejecutar(self, sql, params=None):
        tipos = {bool: 'BOOL', int: 'INT64', float: 'FLOAT64', str: 'STRING'}
        parametros = [self.bq.ScalarQueryParameter(k, tipos[type(v)], v) for k, v in (params or {}).items()]
        config = self.bq.QueryJobConfig(query_parameters=parametros, maximum_bytes_billed=MAX_BYTES_BQ)   # tope de coste por consulta
        job = self.cliente.query(sql, job_config=config)
        return [{k: _limpio(v) for k, v in dict(f.items()).items()} for f in job.result(max_results=MAX_FILAS + 1)]

    def columnas(self, t):
        esquema = self.cliente.get_table(f'{self.proyecto}.{self.dataset}.{t}').schema
        return [{'nombre': c.name, 'tipo': c.field_type} for c in esquema]


def elegir_conector():
    backend = os.environ.get('MCP_BACKEND', 'sqlite').lower()
    if backend == 'sqlite':
        return ConectorSQLite(os.environ.get('CURSO_DB', 'curso.db'))
    if backend == 'bigquery':
        proyecto = os.environ.get('GCP_PROJECT')
        if not proyecto:
            raise RuntimeError('Con MCP_BACKEND=bigquery hace falta GCP_PROJECT (y, si no es curso_master, BQ_DATASET).')
        return ConectorBigQuery(proyecto, os.environ.get('BQ_DATASET', 'curso_master'))
    raise RuntimeError(f'MCP_BACKEND debe ser «sqlite» o «bigquery», no «{backend}».')


# ------------------------------------------------------------------------------------------------ vigilancia del SQL libre
PROHIBIDAS = re.compile(r'\b(insert|update|delete|drop|alter|create|attach|detach|pragma|replace|merge|truncate|grant|revoke|'
                        r'export|call|load|load_extension|readfile|writefile|fts3_tokenizer|vacuum|reindex|begin|commit|execute|set)\b', re.I)


def validar_sql(sql):
    """Devuelve (sql_limpio, None) si es aceptable, o (None, motivo). Es la capa 2: la que NO hay que creerse del todo."""
    if not isinstance(sql, str) or not sql.strip():
        return None, 'Escribe una consulta SELECT.'
    s = sql.strip()
    if '--' in s or '/*' in s or '#' in s:
        return None, 'No se admiten comentarios en la consulta.'
    s = s.rstrip(';').strip()
    if ';' in s:
        return None, 'Solo se admite una sentencia.'
    if not re.match(r'(?is)^(select|with)\b', s):
        return None, 'Solo se admiten consultas que empiecen por SELECT (o WITH).'
    m = PROHIBIDAS.search(s)
    if m:
        return None, f'Palabra no permitida: «{m.group(1).upper()}». Esta herramienta solo lee.'
    ctes = {c.lower() for c in re.findall(r'(?is)(?:\bwith|,)\s*(\w+)\s+as\s*\(', s)}
    for t in re.findall(r'(?is)\b(?:from|join)\s+([`"\[]?[\w.\-]+[`"\]]?)', s):
        t = t.strip('`"[]')
        if '.' in t:
            return None, f'Escribe solo el nombre de la tabla, sin proyecto ni dataset: «{t}».'
        if t.lower() not in TABLAS and t.lower() not in ctes:
            return None, f'La tabla «{t}» no está permitida. Tablas: {", ".join(TABLAS)}.'
    if re.search(r'(?is)\b(?:from|join)\s+\w+(?:\s+(?:as\s+)?\w+)?\s*,', s):
        return None, 'No se admiten listas de tablas separadas por comas: usa JOIN.'
    return s, None


def cualificar(sql, conector):
    """master_clientes -> `proyecto.dataset.master_clientes` en BigQuery (en SQLite no cambia). Solo detrás de FROM/JOIN."""
    return re.sub(r'(?is)\b(from|join)(\s+)(master_\w+)\b', lambda m: f'{m.group(1)}{m.group(2)}{conector.tabla(m.group(3).lower())}', sql)


# ------------------------------------------------------------------------------------------------ servidor
servidor = FastMCP(NOMBRE, instructions=(
    'Consultas de LECTURA sobre la capa master del caso Retail (clientes, facturas, líneas y productos). Para lo habitual usa resumen_cliente, '
    'ultimas_facturas, ventas_por_pais o top_productos; consulta_sql solo si ninguna sirve. Responde con lo que devuelvan las herramientas; '
    'si devuelven «error», díselo al usuario y no inventes datos. No hay nombres ni correos: solo número de cliente y país.'))
_conector = None


def conector():
    global _conector
    if _conector is None:
        _conector = elegir_conector()
    return _conector


def _ejecutar(sql, params=None):
    c = conector()
    try:
        filas = c.ejecutar(cualificar(sql, c), params)
    except Exception as e:                                           # noqa: BLE001 - al LLM le llega el motivo, no un traceback
        return {'error': f'{type(e).__name__}: {str(e)[:300]}', 'motor': c.nombre}
    truncado = len(filas) > MAX_FILAS
    return {'motor': c.nombre, 'filas': filas[:MAX_FILAS], 'n': min(len(filas), MAX_FILAS), 'truncado': truncado}


def _entero(valor, nombre, minimo=1, maximo=None):
    try:
        v = int(valor)
    except (TypeError, ValueError):
        return None, {'error': f'«{nombre}» debe ser un número entero, no «{valor}».'}
    if v < minimo or (maximo is not None and v > maximo):
        return None, {'error': f'«{nombre}» debe estar entre {minimo} y {maximo if maximo is not None else "…"}.'}
    return v, None


@servidor.tool(name='listar_tablas', description='Las tablas de la capa master que se pueden consultar, con su número de filas y el motor (sqlite o bigquery). Empieza por aquí.')
def listar_tablas() -> dict:
    c = conector()
    salida = []
    for t in TABLAS:
        r = _ejecutar(f'SELECT COUNT(*) AS filas FROM {t}')
        salida.append({'tabla': t, 'filas': r['filas'][0]['filas'] if 'filas' in r else r})
    return {'motor': c.nombre, 'tablas': salida}


@servidor.tool(name='describir_tabla', description='Columnas (nombre y tipo) de una tabla master y 3 filas de ejemplo. Argumento: tabla (una de listar_tablas).')
def describir_tabla(tabla: str) -> dict:
    t = str(tabla).strip().lower()
    if t not in TABLAS:
        return {'error': f'La tabla «{tabla}» no existe o no está permitida. Tablas: {", ".join(TABLAS)}.'}
    try:
        columnas = conector().columnas(t)
    except Exception as e:                                           # noqa: BLE001
        return {'error': f'{type(e).__name__}: {str(e)[:300]}'}
    muestra = _ejecutar(f'SELECT * FROM {t} LIMIT 3')
    return {'tabla': t, 'motor': conector().nombre, 'columnas': columnas, 'muestra': muestra.get('filas', muestra)}


@servidor.tool(name='resumen_cliente', description=(
    'Resumen de UN cliente: país, primer y último movimiento, facturas, cancelaciones y monto de ventas válidas. '
    'Úsala cuando pregunten por un cliente concreto. Argumento: cliente_id (número).'))
def resumen_cliente(cliente_id: int) -> dict:
    cid, error = _entero(cliente_id, 'cliente_id', 0)
    if error:
        return error
    r = _ejecutar('''SELECT c.cliente_id, c.pais, c.primer_movimiento, c.ultimo_movimiento,
        COUNT(DISTINCT CASE WHEN f.es_cancelacion = 0 THEN f.factura_id END) AS facturas,
        COUNT(DISTINCT CASE WHEN f.es_cancelacion = 1 THEN f.factura_id END) AS cancelaciones,
        ROUND(SUM(CASE WHEN l.es_venta_valida = 1 THEN l.monto_linea ELSE 0 END), 2) AS monto_ventas
        FROM master_clientes c
        LEFT JOIN master_facturas f ON f.cliente_id = c.cliente_id
        LEFT JOIN master_lineas l ON l.factura_id = f.factura_id
        WHERE c.cliente_id = @cliente_id
        GROUP BY c.cliente_id, c.pais, c.primer_movimiento, c.ultimo_movimiento''', {'cliente_id': cid})
    if 'error' in r:
        return r
    if not r['filas']:
        return {'error': f'No existe el cliente {cid} en master_clientes.'}
    return {'motor': r['motor'], **r['filas'][0]}


@servidor.tool(name='ultimas_facturas', description='Las últimas n facturas de UN cliente (fecha, país, si es cancelación y total de la factura). Argumentos: cliente_id y n (por defecto 5, máximo 20).')
def ultimas_facturas(cliente_id: int, n: int = 5) -> dict:
    cid, error = _entero(cliente_id, 'cliente_id', 0)
    if error:
        return error
    n, error = _entero(n, 'n', 1, 20)
    if error:
        return error
    return _ejecutar('''SELECT f.factura_id, f.fecha_factura, f.pais, f.es_cancelacion, ROUND(SUM(l.monto_linea), 2) AS total
        FROM master_facturas f JOIN master_lineas l ON l.factura_id = f.factura_id
        WHERE f.cliente_id = @cliente_id
        GROUP BY f.factura_id, f.fecha_factura, f.pais, f.es_cancelacion
        ORDER BY f.fecha_factura DESC, f.factura_id DESC LIMIT @n''', {'cliente_id': cid, 'n': n})


@servidor.tool(name='ventas_por_pais', description='Los n países con más ventas válidas: facturas, clientes distintos y monto. Argumento: n (por defecto 5, máximo 50).')
def ventas_por_pais(n: int = 5) -> dict:
    n, error = _entero(n, 'n', 1, 50)
    if error:
        return error
    return _ejecutar('''SELECT f.pais, COUNT(DISTINCT f.factura_id) AS facturas, COUNT(DISTINCT f.cliente_id) AS clientes,
        ROUND(SUM(l.monto_linea), 2) AS monto
        FROM master_facturas f JOIN master_lineas l ON l.factura_id = f.factura_id
        WHERE l.es_venta_valida = 1
        GROUP BY f.pais ORDER BY monto DESC, f.pais LIMIT @n''', {'n': n})


@servidor.tool(name='top_productos', description='Los n productos más vendidos por unidades (solo ventas válidas y productos reales). Argumento: n (por defecto 5, máximo 50).')
def top_productos(n: int = 5) -> dict:
    n, error = _entero(n, 'n', 1, 50)
    if error:
        return error
    return _ejecutar('''SELECT p.codigo_producto, p.descripcion, SUM(l.cantidad) AS unidades, ROUND(SUM(l.monto_linea), 2) AS monto
        FROM master_lineas l JOIN master_productos p ON p.codigo_producto = l.codigo_producto
        WHERE l.es_venta_valida = 1 AND p.es_producto = 1
        GROUP BY p.codigo_producto, p.descripcion ORDER BY unidades DESC, p.codigo_producto LIMIT @n''', {'n': n})


@servidor.tool(name='consulta_sql', description=(
    f'Último recurso: una consulta SELECT propia sobre {", ".join(TABLAS)}. Una sola sentencia, sin comentarios, sin escribir proyecto ni dataset, '
    f'con alias de tabla; devuelve como máximo {MAX_FILAS} filas. Usa antes las otras herramientas. Argumento: sql.'))
def consulta_sql(sql: str) -> dict:
    limpio, motivo = validar_sql(sql)
    if motivo:
        return {'error': motivo, 'rechazada_por': 'vigilancia del SQL'}
    return _ejecutar(limpio)


def main():
    try:
        c = conector()
    except Exception as e:                                           # noqa: BLE001
        print(f'{NOMBRE}: no arranca: {e}', file=sys.stderr)
        sys.exit(1)
    print(f'{NOMBRE}: motor {c.nombre}', file=sys.stderr)               # por stderr: stdout es el canal del protocolo
    servidor.run(transport='stdio')


if __name__ == '__main__':
    main()

Overwriting _trabajo/mcp_datos.py


## 2. El cliente

Un servidor MCP no se usa a mano: lo arranca un cliente (VS Code, un agente...). Esta clase hace de cliente: lanza el servidor como proceso y le pide cosas.
Es la misma mecánica que usa cualquier cliente MCP, en 30 líneas.

In [3]:
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


class ClienteMCP:
    """Lanza un servidor MCP como proceso hijo y habla con él por su entrada y salida estándar (transporte «stdio»).
    Es lo mismo que hacen VS Code, Claude Desktop o un agente: arrancan el servidor, le piden las herramientas y las llaman por nombre."""

    def __init__(self, script, **entorno):
        self.params = StdioServerParameters(command=sys.executable, args=[str(script)], env={**os.environ, **{k: str(v) for k, v in entorno.items()}})

    async def __aenter__(self):
        self._pila = AsyncExitStack()
        self._log = open(TRABAJO / 'servidor.err', 'w', encoding='utf-8')          # lo que el servidor escribe por stderr
        leer, escribir = await self._pila.enter_async_context(stdio_client(self.params, errlog=self._log))
        self.sesion = await self._pila.enter_async_context(ClientSession(leer, escribir))
        await self.sesion.initialize()
        return self

    async def __aexit__(self, *a):
        await self._pila.aclose()
        self._log.close()

    async def herramientas(self):
        return (await self.sesion.list_tools()).tools

    async def llamar(self, nombre, **argumentos):
        r = await self.sesion.call_tool(nombre, argumentos)
        if r.isError:                                                                # el propio protocolo rechazó los argumentos (tipos)
            return {'error_del_protocolo': r.content[0].text}
        return r.structuredContent or json.loads(r.content[0].text)

## 3. Qué ofrece el servidor

Lo primero que hace un cliente es preguntar qué herramientas hay. **La descripción es lo que el LLM lee para decidir cuál usar**: por eso es corta y dice cuándo usarla.

In [4]:
os.environ['CURSO_DB'] = str(BD)
async with ClienteMCP(TRABAJO / 'mcp_datos.py', CURSO_DB=BD, MCP_BACKEND='sqlite') as c:
    for h in await c.herramientas():
        print(f'• {h.name}\n    {h.description}\n    argumentos: {list(h.inputSchema.get("properties", {}))}\n')

• listar_tablas
    Las tablas de la capa master que se pueden consultar, con su número de filas y el motor (sqlite o bigquery). Empieza por aquí.
    argumentos: []

• describir_tabla
    Columnas (nombre y tipo) de una tabla master y 3 filas de ejemplo. Argumento: tabla (una de listar_tablas).
    argumentos: ['tabla']

• resumen_cliente
    Resumen de UN cliente: país, primer y último movimiento, facturas, cancelaciones y monto de ventas válidas. Úsala cuando pregunten por un cliente concreto. Argumento: cliente_id (número).
    argumentos: ['cliente_id']

• ultimas_facturas
    Las últimas n facturas de UN cliente (fecha, país, si es cancelación y total de la factura). Argumentos: cliente_id y n (por defecto 5, máximo 20).
    argumentos: ['cliente_id', 'n']

• ventas_por_pais
    Los n países con más ventas válidas: facturas, clientes distintos y monto. Argumento: n (por defecto 5, máximo 50).
    argumentos: ['n']

• top_productos
    Los n productos más vendidos por unidades (so

## 4. Usarlo con SQLite

Siete llamadas, una por herramienta. Fíjate en que la respuesta siempre dice qué `motor` la ha producido.

In [5]:
CASOS = [('listar_tablas', {}), ('describir_tabla', {'tabla': 'master_clientes'}), ('resumen_cliente', {'cliente_id': 12369}),
         ('ultimas_facturas', {'cliente_id': 12369, 'n': 2}), ('ventas_por_pais', {'n': 3}), ('top_productos', {'n': 3}),
         ('consulta_sql', {'sql': 'SELECT pais, COUNT(*) AS clientes FROM master_clientes GROUP BY pais ORDER BY clientes DESC, pais LIMIT 3'})]


async def ejecutar_casos(**entorno):
    async with ClienteMCP(TRABAJO / 'mcp_datos.py', **entorno) as c:
        return {nombre: await c.llamar(nombre, **args) for nombre, args in CASOS}

resultados_sqlite = await ejecutar_casos(CURSO_DB=BD, MCP_BACKEND='sqlite')
for nombre, r in resultados_sqlite.items():
    print(f'--- {nombre}')
    print(json.dumps(r, ensure_ascii=False, default=str)[:420])

--- listar_tablas
{"motor": "sqlite", "tablas": [{"tabla": "master_clientes", "filas": 150}, {"tabla": "master_facturas", "filas": 1430}, {"tabla": "master_lineas", "filas": 22117}, {"tabla": "master_productos", "filas": 3077}]}
--- describir_tabla
{"tabla": "master_clientes", "motor": "sqlite", "columnas": [{"nombre": "cliente_id", "tipo": "INTEGER"}, {"nombre": "pais", "tipo": "TEXT"}, {"nombre": "primer_movimiento", "tipo": "TEXT"}, {"nombre": "ultimo_movimiento", "tipo": "TEXT"}], "muestra": [{"cliente_id": 12369, "pais": "Austria", "primer_movimiento": "2010-03-08 08:09:00", "ultimo_movimiento": "2010-10-22 13:13:00"}, {"cliente_id": 12425, "pais": "Portug
--- resumen_cliente
{"motor": "sqlite", "cliente_id": 12369, "pais": "Austria", "primer_movimiento": "2010-03-08 08:09:00", "ultimo_movimiento": "2010-10-22 13:13:00", "facturas": 3, "cancelaciones": 0, "monto_ventas": 1511.15}
--- ultimas_facturas
{"motor": "sqlite", "filas": [{"factura_id": "528576", "fecha_factura": "2010-10-

## 5. Intentar romperlo

Un LLM puede equivocarse o ser engañado (por ejemplo, por un texto malicioso dentro de un dato). Estas son consultas que **nunca** deben ejecutarse.
Cada una se frena en una capa distinta: mira cuál.

In [6]:
ATAQUES = {
    'borrar una tabla':               'DROP TABLE master_clientes',
    'modificar datos':                "UPDATE master_clientes SET pais = 'X'",
    'varias sentencias':              'SELECT 1; DELETE FROM master_lineas',
    'comentario para esconder algo':  'SELECT * FROM master_clientes -- WHERE pais = 1',
    'otra tabla (variables del modelo)': 'SELECT * FROM fs_produccion',
    'el catálogo del motor':          'SELECT name FROM sqlite_master',
    'tabla con ruta completa':        'SELECT * FROM main.master_clientes',
    'unir tablas con coma':           'SELECT * FROM master_clientes, master_facturas',
}
async with ClienteMCP(TRABAJO / 'mcp_datos.py', CURSO_DB=BD, MCP_BACKEND='sqlite') as c:
    for nombre, sql in ATAQUES.items():
        r = await c.llamar('consulta_sql', sql=sql)
        print(f'{nombre:<36} → {"RECHAZADA: " + r["error"] if "error" in r else "¡SE EJECUTÓ!"}')
    print()
    r = await c.llamar('consulta_sql', sql='SELECT * FROM master_lineas')
    print(f'SELECT * de 22 117 líneas → devuelve {r["n"]} filas (truncado: {r["truncado"]}): el tope de filas protege al LLM y a su contexto')
    r = await c.llamar('resumen_cliente', cliente_id='abc')
    print('cliente_id = "abc" →', r)

borrar una tabla                     → RECHAZADA: Solo se admiten consultas que empiecen por SELECT (o WITH).
modificar datos                      → RECHAZADA: Solo se admiten consultas que empiecen por SELECT (o WITH).
varias sentencias                    → RECHAZADA: Solo se admite una sentencia.
comentario para esconder algo        → RECHAZADA: No se admiten comentarios en la consulta.
otra tabla (variables del modelo)    → RECHAZADA: La tabla «fs_produccion» no está permitida. Tablas: master_clientes, master_facturas, master_lineas, master_productos.
el catálogo del motor                → RECHAZADA: La tabla «sqlite_master» no está permitida. Tablas: master_clientes, master_facturas, master_lineas, master_productos.
tabla con ruta completa              → RECHAZADA: Escribe solo el nombre de la tabla, sin proyecto ni dataset: «main.master_clientes».
unir tablas con coma                 → RECHAZADA: No se admiten listas de tablas separadas por comas: usa JOIN.

SELECT * de 22 117 lín

**Lo que muestra la prueba**
- Las ocho las frena `validar_sql` (capa 2). Pero **no te fíes de una sola capa**: una expresión regular no es un analizador de SQL.
- Si una se colase, SQLite está abierto en solo lectura (capa 3) y un `UPDATE` fallaría igualmente. Compruébalo tú: quita la vigilancia a mano y prueba (ver el ejercicio 1 al final).
- El último caso lo rechaza el propio protocolo: la herramienta declara `cliente_id: int` y el SDK valida el tipo antes de llamar a tu función.

## 6. BigQuery: el segundo conector

El servidor es **el mismo archivo**. Solo cambia una variable de entorno: `MCP_BACKEND=bigquery` (más `GCP_PROJECT` y `BQ_DATASET`).
Para que haya algo que consultar, primero subimos las cuatro tablas *master* a un dataset de tu proyecto.

> Esta parte necesita un proyecto de Google Cloud y credenciales (notebook 00, apartado 3). Si no las tienes, las celdas dicen «saltada» y puedes seguir.

In [7]:
GCP_PROJECT = os.environ.get('GCP_PROJECT', '')
BQ_DATASET = os.environ.get('BQ_DATASET', 'curso_master')
BQ_LOCATION = os.environ.get('BQ_LOCATION', 'US')
HAY_BQ = False
if GCP_PROJECT:
    try:
        import google.auth
        google.auth.default()
        HAY_BQ = True
    except Exception as e:                                              # noqa: BLE001
        print('Credenciales de Google no disponibles:', str(e)[:140])
print(f'BigQuery: proyecto «{GCP_PROJECT}», dataset «{BQ_DATASET}»' if HAY_BQ else 'BigQuery SALTADO: define GCP_PROJECT y las credenciales (notebook 00, apartado 3).')

BigQuery SALTADO: define GCP_PROJECT y las credenciales (notebook 00, apartado 3).


In [8]:
TABLAS_MASTER = ['master_clientes', 'master_facturas', 'master_lineas', 'master_productos']
if HAY_BQ:
    import pandas as pd
    from google.cloud import bigquery
    bq = bigquery.Client(project=GCP_PROJECT)
    ds = bigquery.Dataset(f'{GCP_PROJECT}.{BQ_DATASET}')
    ds.location = BQ_LOCATION
    bq.create_dataset(ds, exists_ok=True)
    con = sqlite3.connect(BD)
    for t in TABLAS_MASTER:
        df = pd.read_sql(f'SELECT * FROM {t}', con)
        for col in df.columns:                                           # los enteros con huecos llegan como decimales: se devuelven a enteros
            if df[col].dtype == 'float64' and df[col].dropna().apply(float.is_integer).all():
                df[col] = df[col].astype('Int64')
        bq.load_table_from_dataframe(df, f'{GCP_PROJECT}.{BQ_DATASET}.{t}', job_config=bigquery.LoadJobConfig(write_disposition='WRITE_TRUNCATE')).result()
        print(f'{t:<18} {len(df):>6} filas subidas')
    con.close()
else:
    print('saltada')

saltada


## 7. El mismo servidor, otro motor

Mismas siete llamadas. Debe responder lo mismo (salvo el campo `motor`).

In [9]:
if HAY_BQ:
    resultados_bq = await ejecutar_casos(MCP_BACKEND='bigquery', GCP_PROJECT=GCP_PROJECT, BQ_DATASET=BQ_DATASET)
    for nombre, r in resultados_bq.items():
        print(f'--- {nombre}  [{r.get("motor", "?")}]')
        print(json.dumps(r, ensure_ascii=False, default=str)[:300])
else:
    print('saltada')

saltada


## 8. ¿Dan lo mismo?

**Dos conectores que dan respuestas distintas no son dos conectores, son dos bugs.** La comprobación compara cada respuesta ignorando el motor, con una tolerancia de 0,01 en los decimales (los dos motores suman en otro orden).

In [10]:
def igual(a, b):
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(igual(a[k], b[k]) for k in a)
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(igual(x, y) for x, y in zip(a, b))
    if isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool):
        return abs(a - b) <= 0.01
    return a == b

def sin_motor(r):
    r = json.loads(json.dumps(r, default=str))
    r.pop('motor', None)
    if r.get('tablas'):
        r['tablas'] = [dict(t) for t in r['tablas']]
    if r.get('columnas'):                       # los tipos se llaman distinto (INTEGER / INT64): se comparan solo los nombres
        r['columnas'] = [col['nombre'] for col in r['columnas']]
    if r.get('muestra') is not None:
        r['muestra'] = None
    return r

if HAY_BQ:
    for nombre in resultados_sqlite:
        ok = igual(sin_motor(resultados_sqlite[nombre]), sin_motor(resultados_bq[nombre]))
        print(f'{nombre:<20}', 'IGUAL' if ok else 'DISTINTO  <-- revisar')
        assert ok, nombre
    print('\nLos dos conectores responden lo mismo.')
else:
    print('saltada')

saltada


## 9. El tope de coste de BigQuery

En BigQuery se paga por bytes leídos. El servidor fija `maximum_bytes_billed`: si una consulta superara el tope, BigQuery **ni la ejecuta**. Lo forzamos con un tope absurdo de 1 byte.

> **Truco:** BigQuery guarda en caché el resultado de una consulta repetida y esa respuesta **no se factura**, así que no pasaría por el tope. Por eso usamos una consulta que no hemos lanzado antes (`top_productos` con `n=7`). BigQuery responde que hacen falta al menos unos 20 MB (20 971 520 bytes): cobra por mínimos, no por bytes sueltos.

In [11]:
if HAY_BQ:
    async with ClienteMCP(TRABAJO / 'mcp_datos.py', MCP_BACKEND='bigquery', GCP_PROJECT=GCP_PROJECT, BQ_DATASET=BQ_DATASET, MCP_BQ_MAX_BYTES=1) as c:
        r = await c.llamar('top_productos', n=7)
    print(r.get('error', r))
else:
    print('saltada')

saltada


## 10. Limpiar

Si subiste las tablas a BigQuery, bórralas: así no queda nada a tu nombre.

In [12]:
BORRAR_DATASET = True
if HAY_BQ and BORRAR_DATASET:
    bq.delete_dataset(f'{GCP_PROJECT}.{BQ_DATASET}', delete_contents=True, not_found_ok=True)
    print(f'Dataset {BQ_DATASET} borrado.')
else:
    print('Nada que borrar' if not HAY_BQ else 'Dataset conservado (BORRAR_DATASET = False)')

Nada que borrar


## 11. Conectar el servidor a un cliente de verdad

En VS Code (con GitHub Copilot en modo agente), un archivo `.vscode/mcp.json` en tu proyecto:

```json
{
  "servers": {
    "datos-master": {
      "type": "stdio",
      "command": "python",
      "args": ["01-Sesiones/S5-MLOps-y-Operacion/extra-mcp-notebooks/_trabajo/mcp_datos.py"],
      "env": { "CURSO_DB": "04-Proyecto-Base/data/processed/curso.db", "MCP_BACKEND": "sqlite" }
    }
  }
}
```

Para BigQuery cambia `MCP_BACKEND` por `bigquery` y añade `GCP_PROJECT` y `BQ_DATASET`. Pregúntale al agente «¿cuál es el país con más ventas?» y mira qué herramienta elige.

## Ejercicios

1. **Quita una capa.** Edita `validar_sql` para que acepte cualquier cosa y lanza `UPDATE master_clientes SET pais = 'X'`: ¿qué responde SQLite? (Debe fallar por ser de solo lectura.) Devuelve la vigilancia a su sitio.
2. **Añade una herramienta** `clientes_de_un_pais(pais, n)` con parámetros (nunca pegues el texto del país dentro del SQL: es lo que permite la inyección). Pruébala en los dos motores.
3. **Cambia el tope** `MAX_FILAS` a 5 y observa qué cambia en `consulta_sql`.